# Búsqueda Vectorial con FAISS y Azure OpenAI Embeddings

Este notebook implementa búsqueda de similitud vectorial usando FAISS y embeddings generados con Azure OpenAI (ada-002).

referencia: 
https://github.com/facebookresearch/faiss


## Instalación de dependencias

Instalamos las bibliotecas necesarias: FAISS para búsqueda vectorial y OpenAI para Azure OpenAI.


In [5]:
# Instalar dependencias si no están instaladas
!pip install faiss-cpu openai numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.3 MB/s eta 0:00:00m eta 0:00:01-:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.1 MB/s eta 0:00:000:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.1 MB/s eta 0:00:00m eta 0:00:010:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.12.0 requires jax>=0.3.15, which is not installed.
tensorflow 2.12.0 requires libclang>=13.0.0, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
streamlit 1.30.0 requires numpy<2,>=1.19.3, but you have numpy 2.3.5 which is incompatible.
streamlit 1.30.0 requires packaging<24,>=16.8, but you have packaging 24.2 which is incompatible.
con

## Importación de librerías


In [6]:
import faiss
import numpy as np
from openai import AzureOpenAI
from typing import List, Dict, Tuple, Optional
import pickle
import os


## Configuración de Azure OpenAI

Configuración necesaria para conectarse a Azure OpenAI y usar el modelo ada-002 para embeddings.


In [7]:
# Configuración de Azure OpenAI
# Puedes obtener estos valores desde Azure Portal > Azure OpenAI > Keys and Endpoint
AZURE_OPENAI_ENDPOINT = "#"
AZURE_OPENAI_API_KEY = "#"
AZURE_OPENAI_API_VERSION = "2023-05-15"  # O la versión más reciente disponible
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"  # Nombre del deployment de ada-002 en Azure

# Dimensión de los embeddings de ada-002
EMBEDDING_DIMENSION = 1536


## Método 1: Generar embeddings con ada-002

Este método genera embeddings usando el modelo ada-002 de Azure OpenAI.


In [8]:
def generar_embedding(
    texto: str,
    endpoint: str = AZURE_OPENAI_ENDPOINT,
    api_key: str = AZURE_OPENAI_API_KEY,
    api_version: str = AZURE_OPENAI_API_VERSION,
    deployment_name: str = AZURE_OPENAI_EMBEDDING_DEPLOYMENT
) -> np.ndarray:
    """
    Genera un embedding para un texto usando Azure OpenAI ada-002.
    
    Parámetros:
    -----------
    texto : str
        Texto para el cual generar el embedding
    endpoint : str
        Endpoint de Azure OpenAI
    api_key : str
        Clave API de Azure OpenAI
    api_version : str
        Versión de la API de Azure OpenAI
    deployment_name : str
        Nombre del deployment de embeddings en Azure OpenAI
    
    Retorna:
    --------
    np.ndarray
        Vector de embedding de dimensión 1536
    """
    try:
        client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        
        response = client.embeddings.create(
            model=deployment_name,
            input=texto
        )
        
        embedding = np.array(response.data[0].embedding, dtype=np.float32)
        return embedding
        
    except Exception as e:
        print(f"Error al generar embedding: {e}")
        raise


def generar_embeddings_batch(
    textos: List[str],
    endpoint: str = AZURE_OPENAI_ENDPOINT,
    api_key: str = AZURE_OPENAI_API_KEY,
    api_version: str = AZURE_OPENAI_API_VERSION,
    deployment_name: str = AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    batch_size: int = 100
) -> np.ndarray:
    """
    Genera embeddings para múltiples textos en lotes.
    
    Parámetros:
    -----------
    textos : List[str]
        Lista de textos para generar embeddings
    endpoint : str
        Endpoint de Azure OpenAI
    api_key : str
        Clave API de Azure OpenAI
    api_version : str
        Versión de la API de Azure OpenAI
    deployment_name : str
        Nombre del deployment de embeddings en Azure OpenAI
    batch_size : int
        Tamaño del lote para procesar (por defecto 100)
    
    Retorna:
    --------
    np.ndarray
        Matriz de embeddings de forma (n_textos, 1536)
    """
    try:
        client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        
        all_embeddings = []
        
        for i in range(0, len(textos), batch_size):
            batch = textos[i:i + batch_size]
            print(f"Procesando lote {i//batch_size + 1}/{(len(textos)-1)//batch_size + 1}...")
            
            response = client.embeddings.create(
                model=deployment_name,
                input=batch
            )
            
            batch_embeddings = [np.array(item.embedding, dtype=np.float32) for item in response.data]
            all_embeddings.extend(batch_embeddings)
        
        return np.array(all_embeddings, dtype=np.float32)
        
    except Exception as e:
        print(f"Error al generar embeddings en lote: {e}")
        raise


In [9]:
class FAISSSearchIndex:
    """
    Clase para gestionar un índice FAISS con embeddings de Azure OpenAI.
    """
    
    def __init__(
        self,
        dimension: int = EMBEDDING_DIMENSION,
        index_type: str = "L2"
    ):
        """
        Inicializa un índice FAISS.
        
        Parámetros:
        -----------
        dimension : int
            Dimensión de los vectores (1536 para ada-002)
        index_type : str
            Tipo de índice: "L2" (distancia euclidiana) o "cosine" (similitud coseno)
        """
        self.dimension = dimension
        self.index_type = index_type
        self.documents = []  # Almacena los textos originales
        self.metadata = []   # Almacena metadatos opcionales
        
        if index_type == "cosine":
            # Para similitud coseno, normalizamos los vectores
            self.index = faiss.IndexFlatIP(dimension)
            self.normalize = True
        else:
            # Para distancia L2 (euclidiana)
            self.index = faiss.IndexFlatL2(dimension)
            self.normalize = False
    
    def agregar_documentos(
        self,
        textos: List[str],
        embeddings: Optional[np.ndarray] = None,
        metadata: Optional[List[Dict]] = None,
        endpoint: str = AZURE_OPENAI_ENDPOINT,
        api_key: str = AZURE_OPENAI_API_KEY,
        api_version: str = AZURE_OPENAI_API_VERSION,
        deployment_name: str = AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    ):
        """
        Agrega documentos al índice FAISS.
        
        Parámetros:
        -----------
        textos : List[str]
            Lista de textos a agregar
        embeddings : np.ndarray, opcional
            Embeddings pre-generados. Si no se proporciona, se generan automáticamente.
        metadata : List[Dict], opcional
            Metadatos opcionales para cada documento
        endpoint : str
            Endpoint de Azure OpenAI (solo si embeddings no se proporcionan)
        api_key : str
            Clave API de Azure OpenAI (solo si embeddings no se proporcionan)
        api_version : str
            Versión de la API (solo si embeddings no se proporcionan)
        deployment_name : str
            Nombre del deployment (solo si embeddings no se proporcionan)
        """
        if embeddings is None:
            print("Generando embeddings para los documentos...")
            embeddings = generar_embeddings_batch(
                textos,
                endpoint=endpoint,
                api_key=api_key,
                api_version=api_version,
                deployment_name=deployment_name
            )
        
        if embeddings.shape[1] != self.dimension:
            raise ValueError(f"La dimensión de los embeddings ({embeddings.shape[1]}) no coincide con la del índice ({self.dimension})")
        
        if self.normalize:
            faiss.normalize_L2(embeddings)
        
        self.index.add(embeddings)
        self.documents.extend(textos)
        
        if metadata:
            self.metadata.extend(metadata)
        else:
            self.metadata.extend([{}] * len(textos))
        
        print(f"Se agregaron {len(textos)} documentos al índice. Total: {self.index.ntotal}")
    
    def buscar(
        self,
        query: str,
        k: int = 5,
        endpoint: str = AZURE_OPENAI_ENDPOINT,
        api_key: str = AZURE_OPENAI_API_KEY,
        api_version: str = AZURE_OPENAI_API_VERSION,
        deployment_name: str = AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    ) -> List[Tuple[str, float, Dict]]:
        """
        Busca los k documentos más similares a la consulta.
        
        Parámetros:
        -----------
        query : str
            Texto de consulta
        k : int
            Número de resultados a retornar
        endpoint : str
            Endpoint de Azure OpenAI
        api_key : str
            Clave API de Azure OpenAI
        api_version : str
            Versión de la API
        deployment_name : str
            Nombre del deployment
        
        Retorna:
        --------
        List[Tuple[str, float, Dict]]
            Lista de tuplas (documento, score, metadata) ordenadas por similitud
        """
        if self.index.ntotal == 0:
            raise ValueError("El índice está vacío. Agrega documentos primero.")
        
        query_embedding = generar_embedding(
            query,
            endpoint=endpoint,
            api_key=api_key,
            api_version=api_version,
            deployment_name=deployment_name
        )
        
        query_vector = query_embedding.reshape(1, -1).astype(np.float32)
        
        if self.normalize:
            faiss.normalize_L2(query_vector)
        
        distances, indices = self.index.search(query_vector, min(k, self.index.ntotal))
        
        results = []
        for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            if idx != -1:  # -1 indica que no hay resultado
                score = 1 - distance if self.index_type == "L2" else distance
                results.append((
                    self.documents[idx],
                    float(score),
                    self.metadata[idx]
                ))
        
        return results
    
    def buscar_con_embedding(
        self,
        query_embedding: np.ndarray,
        k: int = 5
    ) -> List[Tuple[str, float, Dict]]:
        """
        Busca usando un embedding pre-generado.
        
        Parámetros:
        -----------
        query_embedding : np.ndarray
            Embedding de la consulta
        k : int
            Número de resultados a retornar
        
        Retorna:
        --------
        List[Tuple[str, float, Dict]]
            Lista de tuplas (documento, score, metadata) ordenadas por similitud
        """
        if self.index.ntotal == 0:
            raise ValueError("El índice está vacío. Agrega documentos primero.")
        
        query_vector = query_embedding.reshape(1, -1).astype(np.float32)
        
        if self.normalize:
            faiss.normalize_L2(query_vector)
        
        distances, indices = self.index.search(query_vector, min(k, self.index.ntotal))
        
        results = []
        for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            if idx != -1:
                score = 1 - distance if self.index_type == "L2" else distance
                results.append((
                    self.documents[idx],
                    float(score),
                    self.metadata[idx]
                ))
        
        return results
    
    def guardar(self, ruta_indice: str, ruta_metadata: str):
        """
        Guarda el índice FAISS y los metadatos en disco.
        
        Parámetros:
        -----------
        ruta_indice : str
            Ruta donde guardar el índice FAISS
        ruta_metadata : str
            Ruta donde guardar los documentos y metadatos
        """
        faiss.write_index(self.index, ruta_indice)
        
        with open(ruta_metadata, 'wb') as f:
            pickle.dump({
                'documents': self.documents,
                'metadata': self.metadata,
                'dimension': self.dimension,
                'index_type': self.index_type
            }, f)
        
        print(f"Índice guardado en {ruta_indice}")
        print(f"Metadatos guardados en {ruta_metadata}")
    
    @classmethod
    def cargar(
        cls,
        ruta_indice: str,
        ruta_metadata: str
    ):
        """
        Carga un índice FAISS y metadatos desde disco.
        
        Parámetros:
        -----------
        ruta_indice : str
            Ruta del índice FAISS
        ruta_metadata : str
            Ruta de los documentos y metadatos
        
        Retorna:
        --------
        FAISSSearchIndex
            Instancia del índice cargado
        """
        index = faiss.read_index(ruta_indice)
        
        with open(ruta_metadata, 'rb') as f:
            data = pickle.load(f)
        
        instance = cls(
            dimension=data['dimension'],
            index_type=data['index_type']
        )
        instance.index = index
        instance.documents = data['documents']
        instance.metadata = data['metadata']
        
        print(f"Índice cargado desde {ruta_indice}")
        print(f"Total de documentos: {instance.index.ntotal}")
        
        return instance


### Ejemplo 1: Generar embedding de un texto


In [10]:
texto_ejemplo = "La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo que afecta principalmente a personas mayores."

embedding = generar_embedding(texto_ejemplo)
print(f"Embedding generado. Dimensión: {embedding.shape}")
print(f"Primeros 5 valores: {embedding[:5]}")


Embedding generado. Dimensión: (1536,)
Primeros 5 valores: [-0.01614469 -0.01199597  0.04097611 -0.03131606 -0.01831029]


### Ejemplo 2: Crear índice y agregar documentos


In [11]:
documentos = [
    "La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo que afecta principalmente a personas mayores.",
    "El Alzheimer de inicio temprano puede manifestarse antes de los 65 años y tiene un componente genético más fuerte.",
    "Los biomarcadores pueden detectar la enfermedad de Alzheimer años antes de que aparezcan los síntomas clínicos.",
    "La pérdida de memoria es uno de los primeros síntomas del Alzheimer, especialmente la memoria a corto plazo.",
    "Los tratamientos actuales para el Alzheimer se enfocan en ralentizar la progresión de la enfermedad.",
    "La demencia frontotemporal es diferente del Alzheimer y afecta principalmente a personas más jóvenes.",
    "Los factores de riesgo para el Alzheimer incluyen edad, genética, estilo de vida y factores ambientales.",
    "La investigación sobre el Alzheimer está avanzando en el desarrollo de nuevos tratamientos y métodos de diagnóstico."
]

indice = FAISSSearchIndex(index_type="cosine")

indice.agregar_documentos(documentos)


Generando embeddings para los documentos...
Procesando lote 1/1...
Se agregaron 8 documentos al índice. Total: 8


### Ejemplo 3: Buscar documentos similares


In [12]:
consulta = "¿Cuáles son los síntomas tempranos del Alzheimer?"

resultados = indice.buscar(consulta, k=3)

print(f"Consulta: {consulta}\n")
print("Documentos más similares:")
for i, (doc, score, metadata) in enumerate(resultados, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   Documento: {doc}")


Consulta: ¿Cuáles son los síntomas tempranos del Alzheimer?

Documentos más similares:

1. Score: 0.9050
   Documento: La pérdida de memoria es uno de los primeros síntomas del Alzheimer, especialmente la memoria a corto plazo.

2. Score: 0.8829
   Documento: El Alzheimer de inicio temprano puede manifestarse antes de los 65 años y tiene un componente genético más fuerte.

3. Score: 0.8748
   Documento: La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo que afecta principalmente a personas mayores.


### Ejemplo 4: Agregar documentos con metadatos


In [13]:
indice_con_metadata = FAISSSearchIndex(index_type="cosine")

documentos_metadata = [
    "La pérdida auditiva en mayores está relacionada con un mayor riesgo de deterioro cognitivo.",
    "Los audífonos pueden ayudar a reducir el riesgo de demencia asociado con la pérdida auditiva.",
    "Se recomienda realizar revisiones auditivas periódicas a partir de los 50-60 años."
]

metadata_docs = [
    {"fuente": "artículo_1", "fecha": "2024-01-15", "categoria": "prevención"},
    {"fuente": "artículo_2", "fecha": "2024-02-20", "categoria": "tratamiento"},
    {"fuente": "artículo_3", "fecha": "2024-03-10", "categoria": "recomendaciones"}
]

indice_con_metadata.agregar_documentos(
    textos=documentos_metadata,
    metadata=metadata_docs
)

consulta_metadata = "¿Qué se recomienda para prevenir problemas auditivos?"

resultados_metadata = indice_con_metadata.buscar(consulta_metadata, k=2)

print(f"Consulta: {consulta_metadata}\n")
for i, (doc, score, meta) in enumerate(resultados_metadata, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Documento: {doc}")
    print(f"   Metadatos: {meta}")


Generando embeddings para los documentos...
Procesando lote 1/1...
Se agregaron 3 documentos al índice. Total: 3
Consulta: ¿Qué se recomienda para prevenir problemas auditivos?

1. Score: 0.8798
   Documento: Se recomienda realizar revisiones auditivas periódicas a partir de los 50-60 años.
   Metadatos: {'fuente': 'artículo_3', 'fecha': '2024-03-10', 'categoria': 'recomendaciones'}
2. Score: 0.8700
   Documento: Los audífonos pueden ayudar a reducir el riesgo de demencia asociado con la pérdida auditiva.
   Metadatos: {'fuente': 'artículo_2', 'fecha': '2024-02-20', 'categoria': 'tratamiento'}


### Ejemplo 5: Guardar y cargar índice


In [14]:
ruta_indice = "./indice_faiss.idx"
ruta_metadata = "./metadata_faiss.pkl"

indice.guardar(ruta_indice, ruta_metadata)

indice_cargado = FAISSSearchIndex.cargar(ruta_indice, ruta_metadata)

consulta_cargado = "biomarcadores de Alzheimer"
resultados_cargado = indice_cargado.buscar(consulta_cargado, k=2)

print(f"Consulta: {consulta_cargado}\n")
for i, (doc, score, meta) in enumerate(resultados_cargado, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {doc}")


Índice guardado en ./indice_faiss.idx
Metadatos guardados en ./metadata_faiss.pkl
Índice cargado desde ./indice_faiss.idx
Total de documentos: 8
Consulta: biomarcadores de Alzheimer

1. Score: 0.8931
   Los biomarcadores pueden detectar la enfermedad de Alzheimer años antes de que aparezcan los síntomas clínicos.
2. Score: 0.8611
   La investigación sobre el Alzheimer está avanzando en el desarrollo de nuevos tratamientos y métodos de diagnóstico.


### Ejemplo 6: Búsqueda con embedding pre-generado


In [15]:
consulta_texto = "tratamientos para demencia"
query_embedding = generar_embedding(consulta_texto)

resultados_embedding = indice.buscar_con_embedding(query_embedding, k=3)

print(f"Consulta: {consulta_texto}\n")
for i, (doc, score, meta) in enumerate(resultados_embedding, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {doc}")


Consulta: tratamientos para demencia

1. Score: 0.8671
   Los tratamientos actuales para el Alzheimer se enfocan en ralentizar la progresión de la enfermedad.
2. Score: 0.8526
   La investigación sobre el Alzheimer está avanzando en el desarrollo de nuevos tratamientos y métodos de diagnóstico.
3. Score: 0.8370
   La demencia frontotemporal es diferente del Alzheimer y afecta principalmente a personas más jóvenes.
